# L20 · Ética e interpretabilidad causal

**S20 · Lectura: Cap. 24 · 40–55 min**

## La pregunta

¿Qué objeto estamos explicando y a quién afecta?

## Qué harás

Auditar importancias, proxies y equidad sin sobreinterpretar SHAP.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Una auditoría con advertencias claras.

> **Idea clave:** Explicar una predicción no equivale a explicar el efecto de cambiar una variable.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
rng = setup()
N_REP = R(30, 200); B_BOOT = R(20, 100)
print("réplicas:", N_REP, "· remuestras del bootstrap:", B_BOOT)

In [ ]:
# Auditar antes de interpretar
import matplotlib.pyplot as plt
labels = ['objeto', 'proxy', 'equidad', 'límite']
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.plot(range(len(labels)), [0] * len(labels), "o-", lw=3)
for i, label in enumerate(labels):
    ax.text(i, 0.08, label, ha="center", va="bottom", fontsize=10)
ax.set_xlim(-.35, len(labels) - .65); ax.set_ylim(-.2, .3); ax.axis("off")
ax.set_title("Auditar antes de interpretar")
plt.tight_layout(); plt.show()

## 1. Las tres familias de equidad y su incompatibilidad (§24.2.1)
- **Independencia** (paridad demográfica): $\pi\perp G$ — misma tasa de asignación por grupo.
- **Separación** (equalized odds): $\pi\perp G\mid Y^*$ — mismas tasas de verdaderos y falsos positivos.
- **Suficiencia** (calibración): $Y^*\perp G\mid \pi$ — misma precisión de la decisión por grupo (valor predictivo).

Teorema 24.4: salvo predicción perfecta o tasas base idénticas, **no se pueden cumplir a la vez**. Lo comprobamos en un mundo con tasas base distintas (0,5 y 0,3) y un score bien calibrado dentro de cada grupo.

In [ ]:
def mundo_equidad(n=60000, seed=0):
    r = np.random.default_rng(seed); G = r.binomial(1, .4, n); X = r.normal(size=n)
    riesgo = 1 / (1 + np.exp(-(X + np.where(G == 0, 0.0, -1.0)))); Ystar = r.binomial(1, riesgo)
    return G, riesgo, Ystar                                           # el score ES el riesgo: calibrado dentro de cada grupo
G, s, Ystar = mundo_equidad()
print("tasa base por grupo:", {g: round(float(Ystar[G == g].mean()), 3) for g in (0, 1)})

def metricas_equidad(pi, G, Y):
    f = {}
    for g in (0, 1):
        m = G == g
        f[g] = {"asignación (independencia)": pi[m].mean(), "TPR (separación)": pi[m & (Y == 1)].mean(), "FPR (separación)": pi[m & (Y == 0)].mean(),
                "valor predictivo (suficiencia)": Y[m & (pi == 1)].mean()}
    d = pd.DataFrame(f); d["|brecha|"] = (d[0] - d[1]).abs(); return d
def umbral_para(objetivo, G, s, Y, g1=1):
    grid = np.linspace(.05, .95, 181); m = G == g1
    return min(grid, key=lambda t: abs(objetivo(t, m) ))
# (a) umbral común
pi_a = (s > .5).astype(int)
# (b) igualar la TASA DE ASIGNACIÓN (independencia): el grupo 1 usa el umbral que da la misma tasa que el grupo 0
sel0 = pi_a[G == 0].mean(); t1_b = umbral_para(lambda t, m: (s[m] > t).mean() - sel0, G, s, Ystar)
pi_b = np.where(G == 1, s > t1_b, s > .5).astype(int)
# (c) igualar la TPR (igualdad de oportunidades): el grupo 1 usa el umbral con la misma TPR que el grupo 0
tpr0 = pi_a[(G == 0) & (Ystar == 1)].mean(); t1_c = umbral_para(lambda t, m: (s[m & (Ystar == 1)] > t).mean() - tpr0, G, s, Ystar)
pi_c = np.where(G == 1, s > t1_c, s > .5).astype(int)
tabla = pd.DataFrame({nom: metricas_equidad(p, G, Ystar)["|brecha|"] for nom, p in (("umbral común", pi_a), ("iguala asignación", pi_b), ("iguala TPR", pi_c))})
show(tabla.round(3))

**Mira el resultado.** Mira la diagonal: cada política cierra su brecha. ¿Qué pasa con las demás brechas, y por qué no se arregla con más datos?

**Respuesta:**

Al cerrar una brecha **se abren o persisten otras**: es el teorema, no una limitación del modelo. Con tasas base distintas, el score calibrado produce tasas de asignación, errores y valores predictivos distintos por grupo. Más datos solo estiman mejor esas diferencias. Elegir cuál criterio cumplir es una decisión **normativa** (qué error es más grave aquí), que no se delega al equipo técnico.



## 2. El DGP del Listado 24.1 y un hallazgo sobre el listado (E14)
Mundo del capítulo: $X\sim U(-1,1)^5$, $A$ minoría (30 %), $\tau=2+3X_0-1{,}5A$, propensity $e=0{,}4+0{,}2\tanh X_0-0{,}1A$, $Y=X_1+D\tau+\varepsilon$.

Sabemos la verdad: el efecto medio es **2,0 en la mayoría** y **0,5 en la minoría** (brecha 1,5). Estimamos el CATE con un estimador doblemente robusto (cross-fitting + regresión de las puntuaciones AIPW) y miramos el efecto medio por grupo **con y sin** $A$ entre las covariables.

In [ ]:
ETR = lambda: ExtraTreesRegressor(n_estimators=30, min_samples_leaf=10, max_features=0.8, random_state=0)
def psi_cf(Z, y, d, K=2, seed=SEED):
    n = len(y); psi = np.zeros(n)
    for te in kfold(n, K, seed):
        tr = np.setdiff1d(np.arange(n), te); t1, t0 = tr[d[tr] == 1], tr[d[tr] == 0]
        m1 = ETR().fit(Z[t1], y[t1]).predict(Z[te]); m0 = ETR().fit(Z[t0], y[t0]).predict(Z[te])
        e = np.clip(LogisticRegression(max_iter=500).fit(Z[tr], d[tr]).predict_proba(Z[te])[:, 1], .02, .98)
        psi[te] = m1 - m0 + d[te] * (y[te] - m1) / e - (1 - d[te]) * (y[te] - m0) / (1 - e)
    return psi
def dgp24(n=5000, seed=42, coef_mu0=1.0):
    r = np.random.default_rng(seed); X = r.uniform(-1, 1, (n, 5)); A = r.binomial(1, .3, n); tau = 2 + 3 * X[:, 0] - 1.5 * A
    e = .4 + .2 * np.tanh(X[:, 0]) - .1 * A; D = (r.uniform(size=n) < e).astype(int); Y = coef_mu0 * X[:, 1] + D * tau + r.standard_normal(n) * .5
    return X, A, D, Y, tau
X, A, D, Y, tau = dgp24()
filas = {}
for nom, Z in (("covariables = X (como el listado)", X), ("covariables = X y A", np.column_stack([X, A]))):
    psi = psi_cf(Z, Y, D); cate = ETR().fit(Z, psi).predict(Z)
    filas[nom] = {"CATE medio mayoría": cate[A == 0].mean(), "CATE medio minoría": cate[A == 1].mean(), "brecha estimada": cate[A == 0].mean() - cate[A == 1].mean()}
filas["verdad (EVALUATION)"] = {"CATE medio mayoría": 2.0, "CATE medio minoría": 0.5, "brecha estimada": 1.5}
show(pd.DataFrame(filas).T.round(3))

**Mira el resultado.** En los Listados 24.1 y 24.3 el estimador recibe solo `X`. ¿Por qué no puede ver la brecha, aunque el DGP la contenga?

**Respuesta:**

Porque $A$ no es una covariable del estimador: $\hat\tau$ es una función de $X$ únicamente, y $A$ es independiente de $X$, así que el efecto medio estimado es casi el mismo en los dos grupos (la brecha estimada es ≈ 0, frente a 1,5 reales). Para auditar una brecha por grupo, el atributo **debe entrar al modelo de CATE** (o se estima el CATE por separado en cada grupo). Es una errata del listado del libro.



## 3. ¿Qué objeto se está explicando? (§24.2.3)
Un estimador causal contiene varios modelos: $\hat\mu_Y=E[Y|X]$, $\hat e=E[D|X]$ y $\hat\tau$. El SHAP de $\hat\mu_Y$ dice **qué predice el resultado**, no qué explica el **efecto**. Para explicar $\hat\tau$ se ajusta un **sustituto** sobre sus predicciones y se reporta su fidelidad.

Calculamos valores de Shapley por Monte Carlo (reemplazar las demás variables por filas de un conjunto de fondo y añadir variables en orden aleatorio): estiman la misma cantidad que `shap.TreeExplainer`, sin la dependencia.

In [ ]:
def shapley_mc(pred, Xbg, Xev, M=30, seed=0):
    r = np.random.default_rng(seed); p = Xev.shape[1]; phi = np.zeros((len(Xev), p))
    for _ in range(M):
        perm = r.permutation(p); z = Xbg[r.integers(len(Xbg), size=len(Xev))].copy(); prev = pred(z)
        for j in perm:
            z[:, j] = Xev[:, j]; new = pred(z); phi[:, j] += new - prev; prev = new
    return phi / M
print("shapley_mc listo (importancia = media del valor absoluto por variable)")

### Completa tú: el sustituto del CATE y su fidelidad
`sustituto_cate(Z, cate_hat)` debe: (1) ajustar un `RandomForestRegressor(n_estimators=60, min_samples_leaf=20, random_state=0)` para predecir `cate_hat` a partir de `Z`; (2) calcular el **R² por validación cruzada (5 pliegues)**; (3) devolver `(sustituto_ajustado, r2)`. Si `r2 < 0,8`, la explicación no es fiable.

In [ ]:
# ✏️ Completa tú — sustituto_cate
# Pista: cross_val_score(modelo, Z, cate_hat, cv=5, scoring='r2').mean(); después modelo.fit(Z, cate_hat).
def sustituto_cate(Z, cate_hat):
    """Devuelve (sustituto ajustado, R² de validación cruzada con 5 pliegues)."""
    raise NotImplementedError("completa sustituto_cate")

In [ ]:
FIDEL = {}
def comparar_rankings(coef_mu0, nombre):
    X_, A_, D_, Y_, tau_ = dgp24(5000, 42, coef_mu0); Z = np.column_stack([X_, A_]); nombres = [f"x{i}" for i in range(5)] + ["A"]
    mu_y = ETR().fit(Z, Y_)                                            # modelo de RESULTADO E[Y|X,A]
    psi = psi_cf(Z, Y_, D_); cate_hat = ETR().fit(Z, psi).predict(Z)
    sur, r2 = sustituto_cate(Z, cate_hat)
    ev = Z[:300]
    imp_y = np.abs(shapley_mc(mu_y.predict, Z, ev)).mean(0); imp_t = np.abs(shapley_mc(sur.predict, Z, ev)).mean(0)
    d = pd.DataFrame({"importancia (modelo de RESULTADO)": imp_y, "importancia (sustituto del CATE)": imp_t}, index=nombres)
    d["posición resultado"] = d.iloc[:, 0].rank(ascending=False).astype(int); d["posición CATE"] = d.iloc[:, 1].rank(ascending=False).astype(int)
    print(f"{nombre}: fidelidad del sustituto (R² CV) = {r2:.3f}" + ("" if r2 >= .8 else "  ⚠ < 0,8: la explicación no es fiable"))
    FIDEL[nombre] = r2
    return d.round(3)
r_libro = comparar_rankings(1.0, "DGP del libro (μ0 = 1·x1)"); show(r_libro)
# EVALUATION-CELL: lo que realmente determina τ en el DGP
print("verdad: τ = 2 + 3·x0 − 1,5·A  →  x1 NO afecta al efecto")

In [ ]:
r_amp = comparar_rankings(3.0, "DGP con μ0 más fuerte (μ0 = 3·x1)"); show(r_amp)

**Mira el resultado.** Compara las dos tablas. ¿Qué variable aparece como «importante» para el resultado y es irrelevante para el efecto? ¿Coinciden los primeros puestos?

**Respuesta:**

La variable $x_1$ aparece importante en el modelo de resultado y casi nula en el sustituto del CATE (no entra en $\tau$). Con el DGP del libro el primer puesto puede coincidir (porque $E[Y|X]$ contiene también $e\cdot\tau$, que depende de $x_0$), y se nota sobre todo en el segundo puesto; al reforzar $\mu_0$ el ranking de resultado pasa a liderarlo $x_1$ y la discrepancia salta a la vista. Reportar el primer ranking como «qué determina el efecto» sería un error de categoría, y aun el segundo es asociativo dentro del modelo, no causal sobre el mundo.



## 4. ¿Por qué un t-test sobre los $\hat\tau_i$ da intervalos demasiado estrechos? (§24.3.2–24.3.3)
El IC correcto del efecto medio de un grupo usa la **función de influencia** del estimador (aquí: la desviación de las puntuaciones AIPW $\psi_i$). El enfoque ingenuo usa la desviación de los $\hat\tau_i$ como si fueran observaciones independientes: mide la dispersión del CATE **dentro** del grupo, no el error de estimación. Medimos la cobertura de ambos sobre el efecto verdadero por grupo (2,0 y 0,5).

In [ ]:
POP = {0: 2.0, 1: 0.5}
cub = {"influencia (puntuaciones AIPW)": [], "t-test sobre los τ̂ (ingenuo)": []}; anch = {k: [] for k in cub}
for s_ in range(N_REP):
    X_, A_, D_, Y_, _ = dgp24(1500, 200 + s_); Z = np.column_stack([X_, A_])
    psi = psi_cf(Z, Y_, D_); cate = ETR().fit(Z, psi).predict(Z)
    for g in (0, 1):
        m = A_ == g
        for k, v in (("influencia (puntuaciones AIPW)", psi[m]), ("t-test sobre los τ̂ (ingenuo)", cate[m])):
            se = v.std(ddof=1) / np.sqrt(m.sum()); cub[k].append(abs(v.mean() - POP[g]) < 1.96 * se); anch[k].append(se)
show(pd.DataFrame({"cobertura del IC 95 %": {k: np.mean(v) for k, v in cub.items()}, "error estándar medio": {k: np.mean(v) for k, v in anch.items()}}).round(3))

**Mira el resultado.** ¿Qué ocurre con la cobertura del método ingenuo y qué consecuencia tiene en una auditoría de equidad?

**Respuesta:**

El método ingenuo queda **claramente por debajo de 0,95** (error estándar menor porque ignora el error de estimación de los nuisance): declara como «significativas» brechas que la evidencia no respalda. En una auditoría eso puede justificar decisiones diferenciadas por grupo protegido sobre una base estadística falsa.



## 5. IC de la brecha: influencia frente a bootstrap del pipeline (Listado 24.3)
La brecha es un funcional del modelo. Dos formas válidas: (a) con las puntuaciones AIPW, $\text{se}=\sqrt{v_0/n_0+v_1/n_1}$; (b) **bootstrap del pipeline completo** (re-estimar todo en cada remuestra). El bootstrap es costoso: aquí `B` reducido.

In [ ]:
X_, A_, D_, Y_, tau_ = dgp24(2000, 7); Z = np.column_stack([X_, A_]); psi = psi_cf(Z, Y_, D_)
brecha = psi[A_ == 0].mean() - psi[A_ == 1].mean()
se_if = np.sqrt(psi[A_ == 0].var(ddof=1) / (A_ == 0).sum() + psi[A_ == 1].var(ddof=1) / (A_ == 1).sum())
rb = np.random.default_rng(0); brechas = []
for _ in range(B_BOOT):
    idx = rb.integers(0, len(Y_), len(Y_)); p_ = psi_cf(Z[idx], Y_[idx], D_[idx]); a_ = A_[idx]
    brechas.append(p_[a_ == 0].mean() - p_[a_ == 1].mean())
lo, hi = np.percentile(brechas, [2.5, 97.5])
print(f"brecha estimada: {brecha:.3f}")
print(f"IC por influencia: [{brecha - 1.96 * se_if:.3f}; {brecha + 1.96 * se_if:.3f}]")
print(f"IC por bootstrap del pipeline (B={B_BOOT}): [{lo:.3f}; {hi:.3f}]")
# EVALUATION-CELL
print("brecha verdadera: 1.500")

## 6. Tabla para quienes deciden (Listado 24.2)
Todo número lleva su incertidumbre y se distingue la afirmación **causal** de la **predictiva**.

In [ ]:
def tabla_decisores(psi, A, etiquetas):
    filas = []
    for g, et in etiquetas.items():
        m = A == g; est = psi[m].mean(); se = psi[m].std(ddof=1) / np.sqrt(m.sum())
        filas.append({"Grupo": et, "n": int(m.sum()), "Efecto medio estimado": f"{est:.2f}", "IC 95 %": f"[{est - 1.96 * se:.2f}, {est + 1.96 * se:.2f}]"})
    g0, g1 = psi[A == 0], psi[A == 1]; b = g0.mean() - g1.mean(); sb = np.sqrt(g0.var(ddof=1) / len(g0) + g1.var(ddof=1) / len(g1))
    filas.append({"Grupo": "Brecha (mayoría − minoría)", "n": len(A), "Efecto medio estimado": f"{b:.2f}", "IC 95 %": f"[{b - 1.96 * sb:.2f}, {b + 1.96 * sb:.2f}]"})
    return pd.DataFrame(filas).set_index("Grupo")
tab_dec = tabla_decisores(psi, A_, {0: "Mayoría", 1: "Minoría"}); show(tab_dec)
print("Lectura: estos son efectos CAUSALES estimados del programa (con los supuestos de identificación del estimador), no predicciones del resultado.")

## 7. Supuesto y bitácora

In [ ]:
fila_b = bitacora(
    pregunta="¿Qué variables explican la heterogeneidad del efecto y es significativa la brecha entre grupos?",
    metodo="Shapley (Monte Carlo) del modelo de resultado frente al de un sustituto del CATE; IC por influencia y bootstrap",
    estimando="Efecto medio por grupo y brecha entre grupos",
    numero_ic=f"brecha {brecha:.2f} (IC95% [{brecha - 1.96 * se_if:.2f}; {brecha + 1.96 * se_if:.2f}]); fidelidad del sustituto R²={FIDEL['DGP del libro (μ0 = 1·x1)']:.2f}",
    supuesto=supuesto("falsificable", "contrastado", "la importancia del modelo de resultado NO coincide con la del efecto (x1 relevante para Y, irrelevante para τ); el ranking de resultado no se reporta como «drivers del efecto»"),
    decision="continuar", lab="L20")

---
## Qué queda para el libro
Quedan en el libro: `shap` y `CausalForestDML` de los Listados 24.1–24.3 (aquí se usa un estimador de CATE propio y valores de Shapley por Monte Carlo, que estiman lo mismo que `TreeExplainer`), la auditoría de `gov_transfers` (el dataset no está en el benchmark) y las explicaciones contrafactuales.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*